# 01 — Data Cleaning

This notebook prepares the heart failure readmission dataset for EDA, statistical testing, and modeling.

Cleaning decisions:
- Drop `patient_id` because it is an identifier, not a predictor.
- Check duplicate rows and duplicate patient IDs before removing the ID column.
- Validate categorical and binary columns.
- Convert clinically impossible values to `NaN` using predefined plausible ranges.
- Keep clinically extreme but still plausible values, because they may represent high-risk patients.
- Do **not** impute in this notebook. Imputation is handled later inside the modeling pipeline to avoid data leakage.
- Save the cleaned dataset and a cleaning summary table.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt

from src.data.load import load_raw
from src.data.clean import drop_id, flag_impossible, missing_summary, RANGES
from src.viz.plots import plot_missing
from src.config import PROCESSED_DIR


## Load the raw data and check basic structure

The raw file is loaded first so duplicate rows and duplicate patient IDs can be checked before `patient_id` is dropped.


In [ ]:
raw = load_raw()

raw_shape = pd.DataFrame(
    {
        'rows': [raw.shape[0]],
        'columns': [raw.shape[1]],
        'duplicate_rows': [raw.duplicated().sum()],
        'duplicate_patient_ids': [
            raw['patient_id'].duplicated().sum()
            if 'patient_id' in raw.columns
            else pd.NA
        ],
    }
)

raw_shape


In [ ]:
df = drop_id(raw)
df.shape


## Validate categorical columns

Categorical variables are checked for inconsistent spelling, capitalization, or unexpected labels.


In [ ]:
categorical_cols = ['gender', 'income_level']

category_counts = []
for col in categorical_cols:
    counts = df[col].value_counts(dropna=False).rename_axis('value')
    counts = counts.reset_index(name='count')
    counts.insert(0, 'column', col)
    category_counts.append(counts)

category_counts = pd.concat(category_counts, ignore_index=True)
category_counts


## Validate binary columns

Treatment indicators and the target variable should only contain 0/1 values.


In [ ]:
binary_cols = ['ace_inhibitor', 'beta_blocker', 'diuretic', 'readmitted_30d']

binary_check = []
for col in binary_cols:
    observed_values = sorted(df[col].dropna().unique().tolist())
    binary_check.append(
        {
            'column': col,
            'observed_values': observed_values,
            'valid_binary': set(observed_values).issubset({0, 1}),
            'missing': df[col].isna().sum(),
        }
    )

binary_check = pd.DataFrame(binary_check)
binary_check


## Missingness before cleaning

This shows the missing values already present in the raw dataset after removing `patient_id`.


In [ ]:
before = missing_summary(df)
before


## Plausible clinical ranges

Values outside these ranges are treated as impossible and converted to `NaN`. Values that are extreme but still inside these ranges are retained.


In [ ]:
RANGES


## Flag clinically impossible values

The cleaning step converts impossible values to missing values. These missing values will later be imputed inside the modeling pipeline after the train-test split.


In [ ]:
clean = flag_impossible(df)
after = missing_summary(clean)
after


In [ ]:
# Values newly flagged as impossible during cleaning.
newly_flagged_missing = (
    after['missing'] - before['missing']
).sort_values(ascending=False)

newly_flagged_missing


## Missingness by readmission status

This checks whether missing values are concentrated in either the readmitted or not-readmitted group. Concentrated missingness could suggest that the missing values are not completely random.


In [ ]:
missing_by_outcome = clean.isna().groupby(clean['readmitted_30d']).mean().T * 100
missing_by_outcome = missing_by_outcome.loc[missing_by_outcome.sum(axis=1) > 0]
missing_by_outcome = missing_by_outcome.round(2)
missing_by_outcome


## Cleaning summary table

This table is saved so the technical report can include a clear before-and-after summary of missingness and impossible-value handling.


In [ ]:
cleaning_summary = before.rename(
    columns={'missing': 'missing_before', 'pct': 'pct_before'}
).join(
    after.rename(columns={'missing': 'missing_after', 'pct': 'pct_after'}),
    how='outer',
)

cleaning_summary['newly_flagged_missing'] = (
    cleaning_summary['missing_after'] - cleaning_summary['missing_before']
)

cleaning_summary = cleaning_summary.sort_values(
    ['missing_after', 'newly_flagged_missing'],
    ascending=False,
)

cleaning_summary


## Missingness indicator dataset

The main cleaned file does not include missingness indicators. However, this optional version can be useful if the modeling team wants to test whether missingness itself helps predict readmission.


In [ ]:
indicator_cols = [
    col for col in ['bmi', 'sodium', 'creatinine', 'heart_rate']
    if col in clean.columns and clean[col].isna().any()
]

clean_with_indicators = clean.copy()
for col in indicator_cols:
    clean_with_indicators[f'{col}_missing'] = clean_with_indicators[col].isna().astype(int)

clean_with_indicators.filter(like='_missing').sum().sort_values(ascending=False)


## Visualize remaining missingness


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_missing(clean, ax=ax)
fig.tight_layout()


## Save processed files

`cleaned.csv` is the main modeling dataset. Imputation should still happen later inside the modeling pipeline.


In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean.to_csv(PROCESSED_DIR / 'cleaned.csv', index=False)
cleaning_summary.to_csv(PROCESSED_DIR / 'cleaning_summary.csv')
category_counts.to_csv(PROCESSED_DIR / 'category_value_counts.csv', index=False)
binary_check.to_csv(PROCESSED_DIR / 'binary_validation.csv', index=False)
missing_by_outcome.to_csv(PROCESSED_DIR / 'missingness_by_outcome.csv')
clean_with_indicators.to_csv(
    PROCESSED_DIR / 'cleaned_with_missing_indicators.csv',
    index=False,
)

clean.shape
